# Ridge Regression

## Assignment

We're going back to our other **New York City** real estate dataset. Instead of predicting apartment rents, you'll predict property sales prices.

But not just for condos in Tribeca...

Instead, predict property sales prices for **One Family Dwellings** (`BUILDING_CLASS_CATEGORY` == `'01 ONE FAMILY DWELLINGS'`). 

Use a subset of the data where the **sale price was more than \\$100 thousand and less than $2 million.** 

The [NYC Department of Finance](https://www1.nyc.gov/site/finance/taxes/property-rolling-sales-data.page) has a glossary of property sales terms and NYC Building Class Code Descriptions. The data comes from the [NYC OpenData](https://data.cityofnewyork.us/browse?q=NYC%20calendar%20sales) portal.

- [X] Do train/test split. Use data from January — March 2019 to train. Use data from April 2019 to test.
- [X] Do one-hot encoding of categorical features.
- [X] Do feature selection with `SelectKBest`.
- [ ] Do [feature scaling](https://scikit-learn.org/stable/modules/preprocessing.html).
- [X] Fit a ridge regression model with multiple features.
- [X] Get mean absolute error for the test set.
- [X] As always, commit your notebook to your fork of the GitHub repo.

## My Notes
I made quite a few changes to this notebook when working through it. First, I didn't use pandas_profiling because that causes a lot of dependency issues, especially outside of Colab. Next, I worked on separating my features into numerical and categorical so that I could scale the numerical features with Standard Scaler and use Onehotencoder for my categorical features. 

By implementing that strategy, I decreased the final MAE slightly compared to the original notebook.

In [1]:
%%capture
import sys
# pip install category_encoders==2.*
# pip install fg-data-profiling
# pip install --upgrade jupyter ipywidgets




# Ignore this Numpy warning when using Plotly Express:
# FutureWarning: Method .ptp is deprecated and will be removed in a future version. Use numpy.ptp instead.
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='numpy')

## Load Data

In [2]:



import pandas as pd
import data_profiling
DATA_PATH = 'https://raw.githubusercontent.com/LambdaSchool/DS-Unit-2-Applied-Modeling/master/data/condos/NYC_Citywide_Rolling_Calendar_Sales.csv'


# Read New York City property sales data
df = pd.read_csv(DATA_PATH)

# Change column names: replace spaces with underscores
df.columns = [col.replace(' ', '_') for col in df]

# SALE_PRICE was read as strings.
# Remove symbols, convert to integer
df['SALE_PRICE'] = (
    df['SALE_PRICE']
    .str.replace('$','')
    .str.replace('-','')
    .str.replace(',','')
    .astype(int)
)

In [3]:
# BOROUGH is a numeric column, but arguably should be a categorical feature,
# so convert it from a number to a string
df['BOROUGH'] = df['BOROUGH'].astype(str)

In [4]:
# Reduce cardinality for NEIGHBORHOOD feature

# Get a list of the top 10 neighborhoods
top10 = df['NEIGHBORHOOD'].value_counts()[:10].index

# At locations where the neighborhood is NOT in the top 10, 
# replace the neighborhood with 'OTHER'
df.loc[~df['NEIGHBORHOOD'].isin(top10), 'NEIGHBORHOOD'] = 'OTHER'

In [5]:
df.head()

,BOROUGH,NEIGHBORHOOD,BUILDING_CLASS_CATEGORY,TAX_CLASS_AT_PRESENT,BLOCK,LOT,EASE-MENT,BUILDING_CLASS_AT_PRESENT,ADDRESS,APARTMENT_NUMBER,...,RESIDENTIAL_UNITS,COMMERCIAL_UNITS,TOTAL_UNITS,LAND_SQUARE_FEET,GROSS_SQUARE_FEET,YEAR_BUILT,TAX_CLASS_AT_TIME_OF_SALE,BUILDING_CLASS_AT_TIME_OF_SALE,SALE_PRICE,SALE_DATE
0,1,OTHER,13 CONDOS - ELEVATOR APARTMENTS,2,716,1246,NaN,R4,"447 WEST 18TH STREET, PH12A",PH12A,...,1.0,0.0,1.0,"10,733",1979.0,2007.0,2,R4,0,01/01/2019
1,1,OTHER,21 OFFICE BUILDINGS,4,812,68,NaN,O5,144 WEST 37TH STREET,NaN,...,0.0,6.0,6.0,"2,962",15435.0,1920.0,4,O5,0,01/01/2019
2,1,OTHER,21 OFFICE BUILDINGS,4,839,69,NaN,O5,40 WEST 38TH STREET,NaN,...,0.0,7.0,7.0,"2,074",11332.0,1930.0,4,O5,0,01/01/2019
3,1,OTHER,13 CONDOS - ELEVATOR APARTMENTS,2,592,1041,NaN,R4,"1 SHERIDAN SQUARE, 8C",8C,...,1.0,0.0,1.0,0,500.0,0.0,2,R4,0,01/01/2019
4,1,UPPER EAST SIDE (59-79),15 CONDOS - 2-10 UNIT RESIDENTIAL,2C,1379,1402,NaN,R1,"20 EAST 65TH STREET, B",B,...,1.0,0.0,1.0,0,6406.0,0.0,2,R1,0,01/01/2019


In [6]:
# ('BUILDING_CLASS_CATEGORY' == '01 ONE FAMILY DWELLINGS')

df = df[df['BUILDING_CLASS_CATEGORY'] == '01 ONE FAMILY DWELLINGS']
df = df.drop('EASE-MENT', axis = 1)
df = df.drop('APARTMENT_NUMBER', axis = 1)

## Split Training and Test Data

In [7]:
# - [X] Do train/test split. Use data from January — March 2019 to train. Use data from April 2019 to test.
# 01 - 03 2019 train, 04 2019 test
# property sales prices

train     = df[df['SALE_DATE'].str.contains('0[1-3]/[0-3][0-9]/2019')]
test      = df[df['SALE_DATE'].str.contains('04/[0-3][0-9]/2019')]

target    = 'SALE_PRICE'
high_card = ['BLOCK', 'LOT', 'ADDRESS', 'ZIP_CODE', 'LAND_SQUARE_FEET', 'GROSS_SQUARE_FEET', 'YEAR_BUILT', 'SALE_DATE']
features  = train.columns.drop([target] + high_card)

X_train   = train[features]
y_train   = train[target]
X_test    = test[features]
y_test    = test[target]

In [8]:
train.head()

,BOROUGH,NEIGHBORHOOD,BUILDING_CLASS_CATEGORY,TAX_CLASS_AT_PRESENT,BLOCK,LOT,BUILDING_CLASS_AT_PRESENT,ADDRESS,ZIP_CODE,RESIDENTIAL_UNITS,COMMERCIAL_UNITS,TOTAL_UNITS,LAND_SQUARE_FEET,GROSS_SQUARE_FEET,YEAR_BUILT,TAX_CLASS_AT_TIME_OF_SALE,BUILDING_CLASS_AT_TIME_OF_SALE,SALE_PRICE,SALE_DATE
7,2,OTHER,01 ONE FAMILY DWELLINGS,1,4090,37,A1,1193 SACKET AVENUE,10461.0,1.0,0.0,1.0,"3,404",1328.0,1925.0,1,A1,0,01/01/2019
8,2,OTHER,01 ONE FAMILY DWELLINGS,1,4120,18,A5,1215 VAN NEST AVENUE,10461.0,1.0,0.0,1.0,"2,042",1728.0,1935.0,1,A5,0,01/01/2019
9,2,OTHER,01 ONE FAMILY DWELLINGS,1,4120,20,A5,1211 VAN NEST AVENUE,10461.0,1.0,0.0,1.0,"2,042",1728.0,1935.0,1,A5,0,01/01/2019
42,3,OTHER,01 ONE FAMILY DWELLINGS,1,6809,54,A1,2601 AVENUE R,11229.0,1.0,0.0,1.0,"3,333",1262.0,1925.0,1,A1,0,01/01/2019
44,3,OTHER,01 ONE FAMILY DWELLINGS,1,5495,801,A9,4832 BAY PARKWAY,11230.0,1.0,0.0,1.0,"6,800",1325.0,1930.0,1,A9,550000,01/01/2019


## Check NA Values

In [9]:
train.isnull().sum()

BOROUGH                           0
NEIGHBORHOOD                      0
BUILDING_CLASS_CATEGORY           0
TAX_CLASS_AT_PRESENT              0
BLOCK                             0
LOT                               0
BUILDING_CLASS_AT_PRESENT         0
ADDRESS                           0
ZIP_CODE                          0
RESIDENTIAL_UNITS                 0
COMMERCIAL_UNITS                  0
TOTAL_UNITS                       0
LAND_SQUARE_FEET                  0
GROSS_SQUARE_FEET                 0
YEAR_BUILT                        0
TAX_CLASS_AT_TIME_OF_SALE         0
BUILDING_CLASS_AT_TIME_OF_SALE    0
SALE_PRICE                        0
SALE_DATE                         0
dtype: int64

In [10]:
# - [ ] Do [feature scaling](https://scikit-learn.org/stable/modules/preprocessing.html).
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
import category_encoders as ce

numeric_cols = ['TOTAL_UNITS', 'GROSS_SQUARE_FEET', 'LAND_SQUARE_FEET', 'YEAR_BUILT']


# Setup the preprocessor using data type selectors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), make_column_selector(dtype_include='number')),
        ('cat', ce.OneHotEncoder(use_cat_names = True,), make_column_selector(dtype_include='object'))
    ]
)

# 4. Fit and Transform Training Data
X_train_processed = preprocessor.fit_transform(X_train)

# 5. Transform Test Data ONLY (No Fitting!)
X_test_processed = preprocessor.transform(X_test)

# Optional: Rebuild DataFrames to keep track of columns
cols = preprocessor.get_feature_names_out()
X_train_final = pd.DataFrame(X_train_processed, columns=cols, index=X_train.index)
X_test_final = pd.DataFrame(X_test_processed, columns=cols, index=X_test.index)

In [11]:
X_train_final.head()

,num__RESIDENTIAL_UNITS,num__COMMERCIAL_UNITS,num__TOTAL_UNITS,num__TAX_CLASS_AT_TIME_OF_SALE,cat__BOROUGH_2,cat__BOROUGH_3,cat__BOROUGH_4,cat__BOROUGH_5,cat__BOROUGH_1,cat__NEIGHBORHOOD_OTHER,...,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A9,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A3,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A2,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A0,cat__BUILDING_CLASS_AT_TIME_OF_SALE_S1,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A7,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A4,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A6,cat__BUILDING_CLASS_AT_TIME_OF_SALE_A8,cat__BUILDING_CLASS_AT_TIME_OF_SALE_S0
7,0.092859,-0.154409,-0.086888,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.092859,-0.154409,-0.086888,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.092859,-0.154409,-0.086888,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42,0.092859,-0.154409,-0.086888,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44,0.092859,-0.154409,-0.086888,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
X_train_final.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
num__RESIDENTIAL_UNITS,4094.0,-4.816209e-16,1.000122,-10.769003,0.092859,0.092859,0.092859,0.092859
num__COMMERCIAL_UNITS,4094.0,4.599263e-17,1.000122,-0.154409,-0.154409,-0.154409,-0.154409,12.363406
num__TOTAL_UNITS,4094.0,1.666148e-16,1.000122,-5.476567,-0.086888,-0.086888,-0.086888,10.692471
num__TAX_CLASS_AT_TIME_OF_SALE,4094.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
cat__BOROUGH_2,4094.0,8.524670e-02,0.279283,0.000000,0.000000,0.000000,0.000000,1.000000
cat__BOROUGH_3,4094.0,1.800195e-01,0.384251,0.000000,0.000000,0.000000,0.000000,1.000000
cat__BOROUGH_4,4094.0,4.868100e-01,0.499887,0.000000,0.000000,0.000000,1.000000,1.000000
cat__BOROUGH_5,4094.0,2.401075e-01,0.427201,0.000000,0.000000,0.000000,0.000000,1.000000
cat__BOROUGH_1,4094.0,7.816317e-03,0.088074,0.000000,0.000000,0.000000,0.000000,1.000000
cat__NEIGHBORHOOD_OTHER,4094.0,9.345383e-01,0.247369,0.000000,1.000000,1.000000,1.000000,1.000000


In [13]:
# - [X] Do feature selection with `SelectKBest`.
from sklearn.feature_selection import f_regression, SelectKBest
selector         = SelectKBest(score_func = f_regression, k = 15)
X_train_selected = selector.fit_transform(X_train_final, y_train)
X_test_selected  = selector.transform(X_test_final)

In [14]:
# - [X] Fit a ridge regression model with multiple features.
from sklearn.linear_model import Ridge
model = Ridge(alpha = 15)
model.fit(X_train_final, y_train)
y_pred = model.predict(X_test_final )

In [15]:
pd.DataFrame([X_train_final.columns, model.coef_], index=['Feature', 'Coef']).transpose()

,Feature,Coef
0,num__RESIDENTIAL_UNITS,21316.455232
1,num__COMMERCIAL_UNITS,1420.033151
2,num__TOTAL_UNITS,11800.092223
3,num__TAX_CLASS_AT_TIME_OF_SALE,0.0
4,cat__BOROUGH_2,-564336.529946
5,cat__BOROUGH_3,-498091.934478
6,cat__BOROUGH_4,-585251.675907
7,cat__BOROUGH_5,-596338.620229
8,cat__BOROUGH_1,2244018.760559
9,cat__NEIGHBORHOOD_OTHER,-476230.825462


In [16]:
# - [X] Get mean absolute error for the test set.
import numpy as np
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred)

386766.1850916441

## Future Learnings
- [ ] Instead of `Ridge`, try [`RidgeCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html).
- [ ] Learn more about feature selection:
    - ["Permutation importance"](https://www.kaggle.com/dansbecker/permutation-importance)
    - [scikit-learn's User Guide for Feature Selection](https://scikit-learn.org/stable/modules/feature_selection.html)
    - [mlxtend](http://rasbt.github.io/mlxtend/) library
    - scikit-learn-contrib libraries: [boruta_py](https://github.com/scikit-learn-contrib/boruta_py) & [stability-selection](https://github.com/scikit-learn-contrib/stability-selection)
    - [_Feature Engineering and Selection_](http://www.feat.engineering/) by Kuhn & Johnson.
- [ ] Try [statsmodels](https://www.statsmodels.org/stable/index.html) if you’re interested in more inferential statistical approach to linear regression and feature selection, looking at p values and 95% confidence intervals for the coefficients.
- [ ] Read [_An Introduction to Statistical Learning_](http://faculty.marshall.usc.edu/gareth-james/ISL/ISLR%20Seventh%20Printing.pdf), Chapters 1-3, for more math & theory, but in an accessible, readable way.
- [ ] Try [scikit-learn pipelines](https://scikit-learn.org/stable/modules/compose.html).